# Phase 4 — Legal Risk Agent (first evidence-grounded specialist agent)

**Roadmap reference:** `04_Legal_Risk_Agent.ipynb`

**Scope of this notebook:** the Legal dimension only. Financial and Operational
specialist agents, plus parallel orchestration and consensus across all three,
are Phase 5 (`05_Multi_Agent_Orchestration.ipynb`) — not built here.

**Goal:** for a given contract, probe a fixed set of legal-risk topics
(liability cap, indemnification, termination, confidentiality survival, IP
ownership, governing law/disputes, assignment/change-of-control) using
Phase 3's semantic search scoped to that one document, and ask an LLM to
produce a `Finding` **only** when the retrieved evidence actually supports it.

**Completion gate:** every finding's quoted excerpt is independently
re-checked against the real chunk text before being marked `verified` — a
quote that doesn't genuinely appear in the evidence is flagged
`failed_verification`, not silently trusted. A topic with no supporting
evidence produces no finding at all, rather than a fabricated one.


## 1. Load configuration and the manifest

Requires Phase 3's ChromaDB index to already be built (`chroma_contracts_db/`)
— the agent retrieves evidence from it per-document, it does not re-embed.


In [ ]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.agents import (
    LEGAL_RISK_TOPICS,
    analyze_legal_risk,
    run_phase4_sample,
    run_phase4_batch,
    save_legal_assessment,
)

manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")
print(f"Legal risk topics probed per contract: {list(LEGAL_RISK_TOPICS)}")


Loaded manifest: 36 documents
Legal risk topics probed per contract: ['liability_cap', 'indemnification', 'termination_for_cause', 'termination_for_convenience', 'confidentiality_survival', 'ip_ownership', 'governing_law_disputes', 'assignment_change_of_control']


## 2. Run the agent on a small sample first

Same "sample before scaling to the full corpus" pattern Phase 1 used. One MSA
(`MLA-001`) and one NDA (`MLA-003`) — different document types, so a genuine
gap (a topic the agent honestly finds no evidence for, e.g. an NDA usually has
no liability cap or IP-ownership clause) is expected and fine.


In [5]:
sample_results = run_phase4_sample(["MLA-001", "MLA-003"])


MLA-001: success=True, 7 finding(s) (7 verified, 0 failed_verification)
MLA-003: success=True, 5 finding(s) (5 verified, 0 failed_verification)


## 3. Inspect one result in detail

Every finding below shows its `verification_status` — this is the honesty
check: `verified` means the quoted excerpt was independently found in the
retrieved chunk text, `failed_verification` means the model cited something
that wasn't actually there (kept and flagged, not hidden).


In [6]:
for doc_id, result in sample_results.items():
    print(f"\n{'=' * 70}\n{doc_id} — success={result.success}\n{'=' * 70}")
    for f in result.findings:
        print(f"\n[{f.severity.value}] {f.title}  (confidence={f.confidence}, status={f.verification_status.value})")
        print(f"  rationale: {f.rationale}")
        print(f"  evidence:  {f.evidence_ids} -> {f.evidence_excerpts[0][:160]}...")



MLA-001 — success=True

[HIGH] Limitation of Liability Found  (confidence=0.95, status=verified)
  rationale: The contract explicitly states a cap on total liability, limiting it to the total amount paid by the Client to the Vendor in the preceding twelve months, except for certain breaches.
  evidence:  ['MLA-001-c2'] -> Except for breaches involving confidentiality, intellectual property misuse, or acts of gross negligence or willful misconduct, each Party’s total cumulative li...

[HIGH] Indemnification Obligations  (confidence=0.95, status=verified)
  rationale: The evidence clearly outlines the indemnification obligations of both the Vendor and the Client under the agreement.
  evidence:  ['MLA-001-c2'] -> The Vendor agrees to indemnify and defend the Client against any third-party claims, damages, losses, liabilities, or expenses arising from product defects, wor...

[MEDIUM] Termination for Cause Conditions  (confidence=0.9, status=verified)
  rationale: The evidence specifies 

## 4. Scale to the 15 source-verified documents

The same 15 documents already manually cross-checked against real paragraph
text for Phase 3's retrieval test set — MLA-001..006, FIN-001/002/004,
COM-001/002, NEG-003, GOV-004, OPS-003/004 — across MSAs, NDAs, rate cards,
billing schedules, pricing tables, a redlined negotiation, renewal notes, and
service reports. Running the legal agent on all 15 (instead of just the
MSA/NDA sample) surfaces genuine per-doc-type coverage differences: a rate
card or service report should legitimately produce far fewer legal findings
than an MSA, and that's expected, not a bug.

`run_phase4_batch()` saves each doc's individual report (same as
`run_phase4_sample`) **and** writes one aggregate
`phase4_legal_risk_summary.json` — per-doc finding counts, verification
status breakdown, and which topics were/weren't addressed — so the 15
results can be compared side by side instead of opening 15 files.

In [10]:
BATCH_DOC_IDS = [
    "MLA-001", "MLA-002", "MLA-003", "MLA-004", "MLA-005", "MLA-006",
    "FIN-001", "FIN-002", "FIN-004",
    "COM-001", "COM-002",
    "NEG-003",
    "GOV-004",
    "OPS-003", "OPS-004",
]
batch_summary = run_phase4_batch(BATCH_DOC_IDS)


MLA-001: success=True, 7 finding(s) (7 verified, 0 failed_verification)
MLA-002: success=True, 7 finding(s) (7 verified, 0 failed_verification)
MLA-003: success=True, 5 finding(s) (5 verified, 0 failed_verification)
MLA-004: success=True, 5 finding(s) (5 verified, 0 failed_verification)
MLA-005: success=True, 1 finding(s) (1 verified, 0 failed_verification)
MLA-006: success=True, 1 finding(s) (1 verified, 0 failed_verification)
FIN-001: success=True, 2 finding(s) (2 verified, 0 failed_verification)
FIN-002: success=True, 2 finding(s) (2 verified, 0 failed_verification)
FIN-004: success=True, 0 finding(s) (0 verified, 0 failed_verification)
COM-001: success=True, 2 finding(s) (2 verified, 0 failed_verification)
COM-002: success=True, 3 finding(s) (3 verified, 0 failed_verification)
NEG-003: success=True, 6 finding(s) (6 verified, 0 failed_verification)
GOV-004: success=True, 2 finding(s) (2 verified, 0 failed_verification)
OPS-003: success=True, 0 finding(s) (0 verified, 0 failed_verifi

## Honest status of Phase 4 at this point

- **Built:** `analyze_legal_risk(doc_id)` — 8 fixed legal topics, each grounded
  in Phase 3 retrieval scoped to that one document, each finding independently
  re-verified against the actual retrieved chunk text before being trusted.
- **Not built yet (Phase 5):** Financial agent, Operational agent, parallel
  execution across all three, consensus/composite risk scoring
  (`RISK_WEIGHTS` in `config.py` is defined but unused until then).
- **Known limitation carried over from Phase 3:** a topic can only be found if
  Phase 3's retrieval actually surfaces the right chunk for it — the
  confidentiality-definition retrieval gap documented in
  `artifacts/evaluations/phase3_retrieval_test_set.json` applies here too.
- **Not real yet:** anything in `PWC_CONTRACTIQ_PROJECT_IK/Research/Week-5..8
  Assignment.ipynb` (`ContractIntelligenceSystem`, `ContractRiskOrchestrator`,
  `ContractKnowledgeGraph`). That is a separate, earlier course-assignment
  project and is not wired into this capstone rebuild — nothing from it is
  reused here without being rebuilt and re-verified against this project's
  own models/config/retrieval layer.
